# Emergency Supply Delivery Planning Using Search Algorithms
**AIML Practical Assignment 2** — Ashmit Amit Budhe · PRN 202401040047 · A12

Run **Runtime → Run all** in Google Colab. Python standard library only.

## Contents
1. **Uninformed:** BFS, DFS, UCS, Depth-Limited, Iterative Deepening, Bidirectional.
2. **Informed:** Greedy Best-First, A*, IDA*, RBFS, SMA*.
3. **Local:** Hill Climbing, Simulated Annealing, Local Beam, Genetic Algorithm.
4. **Constraint Satisfaction:** Backtracking, Forward Checking, AC-3 + Backtracking, Min-Conflicts.

These are all the methods listed in our discussion, not an exhaustive list of every search variant. Forward checking and AC-3 are inference techniques; Min-Conflicts is local search for CSPs. Route algorithms share a road map. Local algorithms share a hospital visit-order objective. CSP methods share a dispatch-slot assignment problem. Compare within each task, not across different objectives.

In [1]:
import heapq, itertools, math, time
from collections import deque

# Coordinate units and road costs are minutes. Every road cost >= straight-line distance,
# making Euclidean distance an admissible, consistent A* heuristic.
pos = {'Depot':(0,0),'A':(1,2),'B':(2,0),'C':(3,3),'D':(4,1),
       'Hospital North':(5,4),'Hospital Central':(6,2),
       'E':(7,0),'Hospital East':(9,1)}
roads = [('Depot','A',3),('Depot','B',3),('A','C',3),('A','D',5),
         ('B','D',3),('B','E',8),('C','Hospital North',3),('C','D',3),
         ('D','Hospital Central',3),('D','E',4),
         ('Hospital North','Hospital Central',3),
         ('Hospital Central','Hospital East',4),('E','Hospital East',3)]
graph={v:{} for v in pos}
for a,b,w in roads:
    assert w>=math.dist(pos[a],pos[b]), (a,b)
    graph[a][b]=w; graph[b][a]=w

def path_cost(path):
    return sum(graph[a][b] for a,b in zip(path,path[1:])) if path else math.inf

import random, statistics, io, contextlib
START, GOAL = 'Depot', 'Hospital East'
def heuristic(a,b): return math.dist(pos[a],pos[b])
route_methods={}; local_methods={}; csp_methods={}
def register_route(name,fn): route_methods[name]=fn


## 1. Uninformed search — choose a route
No estimate of remaining distance is used. BFS, iterative deepening and bidirectional BFS minimize road count here; UCS minimizes travel minutes. DFS and depth-limited search make no optimality guarantee.

### 1.1 Breadth-First Search (BFS)
Queue-based, level-by-level expansion. Stops at the first goal reached.

In [2]:
def bfs(start,goal):
    queue=deque([(start,[start])]); seen={start}; expanded=0
    while queue:
        node,path=queue.popleft(); expanded+=1
        if node==goal: return path,expanded
        for nxt in graph[node]:
            if nxt not in seen:
                seen.add(nxt); queue.append((nxt,path+[nxt]))
    return None,expanded

register_route('BFS',bfs)

### 1.2 Depth-First Search (DFS)
A stack follows one branch deeply before trying another. Visited nodes prevent cycles.

In [3]:
def dfs(start,goal):
    stack=[(start,[start])]; seen=set(); expanded=0
    while stack:
        node,path=stack.pop()
        if node in seen: continue
        seen.add(node); expanded+=1
        if node==goal: return path,expanded
        for nxt in reversed(list(graph[node])):
            if nxt not in seen: stack.append((nxt,path+[nxt]))
    return None,expanded
register_route('DFS',dfs)

### 1.3 Uniform Cost Search (UCS)
Expand the lowest accumulated travel cost g(n). This common engine will also support A*.

In [4]:
def weighted_search(start,goal,h):
    queue=[(h(start,goal),0,start,[start])]; best={start:0}; expanded=0
    while queue:
        _,cost,node,path=heapq.heappop(queue)
        if cost!=best[node]: continue
        expanded+=1
        if node==goal: return path,expanded
        for nxt,w in graph[node].items():
            new=cost+w
            if new<best.get(nxt,math.inf):
                best[nxt]=new
                heapq.heappush(queue,(new+h(nxt,goal),new,nxt,path+[nxt]))
    return None,expanded

def ucs(start,goal): return weighted_search(start,goal,lambda a,b:0)
register_route('UCS',ucs)

### 1.4 Depth-Limited Search (DLS)
DFS with a maximum number of roads. This example uses a limit of 3; a smaller limit can miss a reachable destination.

In [5]:
def depth_limited(start,goal,limit=3):
    expanded=0
    def visit(path,left):
        nonlocal expanded
        expanded+=1; node=path[-1]
        if node==goal: return path
        if left==0: return None
        for nxt in graph[node]:
            if nxt not in path:
                result=visit(path+[nxt],left-1)
                if result: return result
        return None
    return visit([start],limit),expanded
register_route('DLS (limit 3)',depth_limited)

### 1.5 Iterative Deepening DFS (IDDFS)
Repeat DLS with increasing limits. Counts include repeated work across iterations.

In [6]:
def iddfs(start,goal):
    expanded=0
    for limit in range(len(graph)):
        path,count=depth_limited(start,goal,limit); expanded+=count
        if path: return path,expanded
    return None,expanded
register_route('IDDFS',iddfs)

### 1.6 Bidirectional BFS
Search from both endpoints on this undirected map, expanding complete layers and joining at a meeting node. Optimizes road count, not weighted minutes.

In [7]:
def bidirectional(start,goal):
    if start==goal: return [start],0
    fronts=[{start},{goal}]; paths=[{start:[start]},{goal:[goal]}]; expanded=0
    while all(fronts):
        side=0 if len(fronts[0])<=len(fronts[1]) else 1
        next_front=set(); meetings=[]
        for node in sorted(fronts[side]):
            expanded+=1
            for nxt in graph[node]:
                if nxt not in paths[side]:
                    paths[side][nxt]=paths[side][node]+[nxt]; next_front.add(nxt)
                if nxt in paths[1-side]: meetings.append(nxt)
        if meetings:
            meet=min(meetings,key=lambda x:len(paths[0][x])+len(paths[1][x]))
            return paths[0][meet]+list(reversed(paths[1][meet]))[1:],expanded
        fronts[side]=next_front
    return None,expanded
register_route('Bidirectional BFS',bidirectional)

## 2. Informed search — use a heuristic
The heuristic is straight-line distance in coordinate units calibrated as minutes. Each road cost is at least that distance, so it never overestimates the minimum remaining travel time. This is simulated geometry, not real traffic prediction.

### 2.1 Greedy Best-First Search
Uses only estimated remaining cost h(n). It can reach the goal quickly but is not guaranteed to choose the fastest route.

In [8]:
def greedy(start,goal):
    queue=[(heuristic(start,goal),start,[start])]; seen=set(); expanded=0
    while queue:
        _,node,path=heapq.heappop(queue)
        if node in seen: continue
        seen.add(node); expanded+=1
        if node==goal: return path,expanded
        for nxt in graph[node]:
            if nxt not in seen: heapq.heappush(queue,(heuristic(nxt,goal),nxt,path+[nxt]))
    return None,expanded
register_route('Greedy Best-First',greedy)

### 2.2 A* Search
Uses f(n) = g(n) + h(n): actual travel time plus estimated remaining time.

In [9]:
def astar(start,goal): return weighted_search(start,goal,heuristic)
register_route('A*',astar)

### 2.3 Iterative Deepening A* (IDA*)
Depth-first search with an increasing f-cost threshold. The next threshold is the smallest exceeded f-value.

In [10]:
def ida_star(start,goal):
    bound=heuristic(start,goal); expanded=0
    def visit(path,cost,bound):
        nonlocal expanded
        f=cost+heuristic(path[-1],goal)
        if f>bound: return None,f
        expanded+=1
        if path[-1]==goal: return path,f
        next_bound=math.inf
        for nxt,w in graph[path[-1]].items():
            if nxt in path: continue
            result,exceeded=visit(path+[nxt],cost+w,bound)
            if result: return result,exceeded
            next_bound=min(next_bound,exceeded)
        return None,next_bound
    while True:
        path,next_bound=visit([start],0,bound)
        if path or next_bound==math.inf: return path,expanded
        bound=next_bound
register_route('IDA*',ida_star)

### 2.4 Recursive Best-First Search (RBFS)
Recursively follows the best f-value while remembering an alternative bound. Backs up costs when returning to a parent.

In [11]:
def rbfs(start,goal):
    expanded=0
    def visit(path,g,f,limit):
        nonlocal expanded
        expanded+=1; node=path[-1]
        if node==goal: return path,f
        children=[]
        for nxt,w in graph[node].items():
            if nxt not in path:
                ng=g+w; children.append([max(f,ng+heuristic(nxt,goal)),ng,path+[nxt]])
        if not children: return None,math.inf
        while True:
            children.sort(key=lambda c:(c[0],c[2]))
            best=children[0]
            if best[0]>limit or best[0]==math.inf: return None,best[0]
            alternative=children[1][0] if len(children)>1 else math.inf
            result,new_f=visit(best[2],best[1],best[0],min(limit,alternative))
            best[0]=new_f
            if result: return result,new_f
    result,_=visit([start],0,heuristic(start,goal),math.inf)
    return result,expanded
register_route('RBFS',rbfs)

### 2.5 Simplified Memory-Bounded A* (SMA*)
Educational tree-search implementation with a fixed budget of node records, worst-leaf removal, backed-up f-values and regeneration of forgotten branches. Ancestor cycles are rejected. Forgotten-successor scalar bookkeeping adds overhead beyond the record budget. A budget too small to retain a solution path can cause failure.

In [12]:
def sma_star(start,goal,memory_limit=20):
    class Node:
        def __init__(self,path,g,f,parent=None):
            self.path=path; self.g=g; self.f=f; self.parent=parent
            self.options=None; self.children={}; self.forgotten={}
    root=Node([start],0,heuristic(start,goal)); records=[root]; expanded=0
    def backup(node):
        while node:
            if node.options is not None:
                values=[c.f for c in node.children.values()]+list(node.forgotten.values())
                node.f=max(node.g+heuristic(node.path[-1],goal),min(values,default=math.inf))
            node=node.parent
    for _ in range(10000):
        node=root
        while True:
            if node.f==math.inf: return None,expanded
            if node.path[-1]==goal: return node.path,expanded
            if node.options is None:
                expanded+=1
                node.options=[x for x in graph[node.path[-1]] if x not in node.path]
                for nxt in node.options:
                    g=node.g+graph[node.path[-1]][nxt]
                    node.forgotten[nxt]=max(node.f,g+heuristic(nxt,goal))
                backup(node)
                break # Re-select globally after the backed-up estimate changes.
            choices=[(c.f,k,True) for k,c in node.children.items()]
            choices += [(f,k,False) for k,f in node.forgotten.items()]
            f,nxt,present=min(choices)
            if f==math.inf: backup(node); break
            if present: node=node.children[nxt]; continue
            g=node.g+graph[node.path[-1]][nxt]
            child=Node(node.path+[nxt],g,f,node)
            del node.forgotten[nxt]; node.children[nxt]=child; records.append(child)
            if len(records)>memory_limit:
                # Never delete an ancestor: only non-root leaves are eligible.
                leaves=[r for r in records if r.parent and not r.children]
                worst=max(leaves,key=lambda r:(r.f,-len(r.path),r.path))
                parent=worst.parent; key=worst.path[-1]
                parent.forgotten[key]=worst.f
                del parent.children[key]; records.remove(worst); backup(parent)
                if worst is child:
                    # A full-memory path cannot be extended; mark this branch unavailable.
                    if len(node.path)>=memory_limit:
                        node.forgotten[nxt]=math.inf; backup(node)
                    break
            break # Re-select globally after generating or forgetting a node.
    raise RuntimeError('SMA* regeneration limit reached; increase memory budget')
register_route('SMA* (20 records)',sma_star)

### 2.6 Route-search performance comparison
All eleven route methods use the same endpoints and map. Median runtime over five runs reduces some noise; tiny-instance timing is still not a benchmark. Expansion counts include revisits where applicable.

In [13]:
route_results=[]
for name,fn in route_methods.items():
    timings=[]
    for _ in range(5):
        t=time.perf_counter(); path,expanded=fn(START,GOAL)
        timings.append((time.perf_counter()-t)*1000)
    route_results.append((name,path,path_cost(path),expanded,statistics.median(timings)))
print(f'{"Method":<24}{"Roads":>7}{"Minutes":>10}{"Expanded":>11}{"Median ms":>12}')
for name,path,cost,count,ms in route_results:
    print(f'{name:<24}{len(path)-1 if path else "—":>7}{cost:>10}{count:>11}{ms:>12.4f}')
    print('   ',' → '.join(path) if path else 'No route found')

Method                    Roads   Minutes   Expanded   Median ms
BFS                           3        14          9      0.0031
    Depot → B → E → Hospital East
DFS                           8        29          9      0.0041
    Depot → A → C → Hospital North → Hospital Central → D → B → E → Hospital East
UCS                           4        13          9      0.0061
    Depot → B → D → Hospital Central → Hospital East
DLS (limit 3)                 3        14         19      0.0043
    Depot → B → E → Hospital East
IDDFS                         3        14         30      0.0064
    Depot → B → E → Hospital East
Bidirectional BFS             3        14          4      0.0046
    Depot → B → E → Hospital East
Greedy Best-First             3        14          4      0.0028
    Depot → B → E → Hospital East
A*                            4        13          8      0.0068
    Depot → B → D → Hospital Central → Hospital East
IDA*                          4        13         36     

## 3. Local search — optimize hospital visit order
One vehicle leaves the depot, visits all hospitals, and returns. A state is a permutation of hospitals. Swap neighbors change two visits. UCS computes the travel cost between stops. This small example has only six orders, so exhaustive enumeration provides a reference optimum; it is not part of the four local methods.

In [14]:
hospitals=['Hospital North','Hospital Central','Hospital East']
points=['Depot']+hospitals
leg={(a,b):path_cost(ucs(a,b)[0]) for a in points for b in points if a!=b}
def tour_cost(order):
    stops=['Depot',*order,'Depot']
    return sum(leg[a,b] for a,b in zip(stops,stops[1:]))
def neighbors(order):
    result=[]
    for i,j in itertools.combinations(range(len(order)),2):
        x=list(order); x[i],x[j]=x[j],x[i]; result.append(tuple(x))
    return result
initial=('Hospital East','Hospital North','Hospital Central')
reference=min(itertools.permutations(hospitals),key=tour_cost)
print('Initial tour:',initial,'Cost:',tour_cost(initial),'Reference optimum:',tour_cost(reference))

Initial tour: ('Hospital East', 'Hospital North', 'Hospital Central') Cost: 32 Reference optimum: 29


### 3.1 Hill Climbing (steepest ascent / cost descent)
Evaluate all swap neighbors and take the best strict improvement. Can stop at a local optimum.

In [15]:
def hill_climbing(initial):
    current=initial; evaluations=1
    while True:
        candidates=neighbors(current); evaluations+=len(candidates)
        best=min(candidates,key=tour_cost)
        if tour_cost(best)>=tour_cost(current): return current,evaluations
        current=best
local_methods['Hill Climbing']=hill_climbing

### 3.2 Simulated Annealing
Sometimes accepts a worse tour with probability exp(-increase / temperature). Temperature decreases each step; retain the best tour encountered.

In [16]:
def simulated_annealing(initial,seed=42):
    rng=random.Random(seed); current=best=initial; temperature=10.; evaluations=1
    for _ in range(200):
        trial=rng.choice(neighbors(current)); evaluations+=1
        delta=tour_cost(trial)-tour_cost(current)
        if delta<=0 or rng.random()<math.exp(-delta/temperature): current=trial
        if tour_cost(current)<tour_cost(best): best=current
        temperature*=0.97
    return best,evaluations
local_methods['Simulated Annealing']=simulated_annealing

### 3.3 Local Beam Search
Keep k candidate orders together, generate all their neighbors, and keep the best k distinct orders.

In [17]:
def local_beam(initial,k=2):
    beam=[initial]; evaluations=1
    for _ in range(20):
        generated=[x for state in beam for x in neighbors(state)]
        evaluations+=len(generated)
        next_beam=sorted(set(beam+generated),key=lambda x:(tour_cost(x),x))[:k]
        if next_beam==beam: break
        beam=next_beam
    return min(beam,key=tour_cost),evaluations
local_methods['Local Beam Search']=local_beam

### 3.4 Genetic Algorithm
A population of permutations uses tournament selection, order-preserving crossover, swap mutation and elitism. Fixed seeds make the demonstration reproducible.

In [18]:
def genetic_algorithm(initial,seed=42,population_size=12,generations=30):
    rng=random.Random(seed)
    population=[initial]+[tuple(rng.sample(list(initial),len(initial))) for _ in range(population_size-1)]
    evaluations=0
    def select(): return min(rng.sample(population,3),key=tour_cost)
    for _ in range(generations):
        evaluations+=len(population)
        next_population=[min(population,key=tour_cost)]
        while len(next_population)<population_size:
            a,b=select(),select(); cut=rng.randrange(1,len(a))
            child=a[:cut]+tuple(x for x in b if x not in a[:cut])
            if rng.random()<0.25: child=rng.choice(neighbors(child))
            next_population.append(child)
        population=next_population
    evaluations+=len(population)
    return min(population,key=tour_cost),evaluations
local_methods['Genetic Algorithm']=genetic_algorithm

In [19]:
local_results=[]
print(f'{"Method":<25}{"Tour minutes":>14}{"Candidates scored*":>21}{"ms":>10}')
for name,fn in local_methods.items():
    t=time.perf_counter(); tour,count=fn(initial); ms=(time.perf_counter()-t)*1000
    local_results.append((name,tour,tour_cost(tour),count,ms))
    print(f'{name:<25}{tour_cost(tour):>14}{count:>21}{ms:>10.4f}');print('   ',tour)
print('*Reported generated/scored candidate count, not every internal cost-function call.')

Method                     Tour minutes   Candidates scored*        ms
Hill Climbing                        29                    7    0.0194
    ('Hospital North', 'Hospital East', 'Hospital Central')
Simulated Annealing                  29                  201    0.7260
    ('Hospital East', 'Hospital Central', 'Hospital North')
Local Beam Search                    29                   16    0.0262
    ('Hospital Central', 'Hospital East', 'Hospital North')
Genetic Algorithm                    29                  372    2.9696
    ('Hospital Central', 'Hospital East', 'Hospital North')
*Reported generated/scored candidate count, not every internal cost-function call.


## 4. Constraint satisfaction — schedule dispatch slots
To demonstrate AC-3 accurately, this expanded notebook uses a **binary dispatch-slot CSP**, replacing the earlier simplified van-capacity assignment. Variables are hospital deliveries; values are dispatch times (minutes after the depot opens). Each delivery is a separate trip with a fresh supply load. A single loading bay needs five minutes between dispatches. Travel time + dispatch time must meet each hospital deadline. No multi-stop arrival prediction or total van-capacity scheduling is claimed.

Unary constraints filter individual slot domains. Binary constraints require any two dispatches to be at least five minutes apart. All four methods solve this same CSP. AC-3 alone may not finish a CSP, so it is followed by backtracking. Min-Conflicts has a step limit and cannot certify infeasibility.

In [20]:
slots=[0,5,10]
travel={h:path_cost(ucs('Depot',h)[0]) for h in hospitals}
deadlines={'Hospital North':19,'Hospital Central':14,'Hospital East':13}
def make_domains():
    return {h:[s for s in slots if s+travel[h]<=deadlines[h]] for h in hospitals}
def compatible(a,x,b,y): return a==b or abs(x-y)>=5
def valid_assignment(a):
    return len(a)==len(hospitals) and all(a[h] in make_domains()[h] for h in hospitals) and all(
        compatible(x,a[x],y,a[y]) for x,y in itertools.combinations(hospitals,2))
print('Travel times:',travel,' / Deadlines:',deadlines,' / Allowed dispatch slots:',make_domains())

Travel times: {'Hospital North': 9, 'Hospital Central': 9, 'Hospital East': 13}  / Deadlines: {'Hospital North': 19, 'Hospital Central': 14, 'Hospital East': 13}  / Allowed dispatch slots: {'Hospital North': [0, 5, 10], 'Hospital Central': [0, 5], 'Hospital East': [0]}


### 4.1 Plain Backtracking Search
Assign one variable at a time in hospital order. On a conflict, undo the assignment and try another slot.

In [21]:
def solve_backtracking(domains=None):
    domains=make_domains() if domains is None else {h:list(v) for h,v in domains.items()}
    assignment={}; attempts=0
    def visit():
        nonlocal attempts
        if len(assignment)==len(hospitals): return dict(assignment)
        h=next(x for x in hospitals if x not in assignment)
        for value in domains[h]:
            attempts+=1
            if all(compatible(h,value,k,v) for k,v in assignment.items()):
                assignment[h]=value
                result=visit()
                if result is not None: return result
                del assignment[h]
        return None
    return visit(),attempts
csp_methods['Backtracking']=solve_backtracking

### 4.2 Backtracking with Forward Checking
After an assignment, delete conflicting values from every unassigned domain. An empty domain triggers immediate backtracking. Uses minimum remaining values (MRV) to select the next variable.

In [22]:
def forward_checking():
    attempts=0
    def visit(assignment,domains):
        nonlocal attempts
        if len(assignment)==len(hospitals): return assignment
        h=min((x for x in hospitals if x not in assignment),key=lambda x:len(domains[x]))
        for value in domains[h]:
            attempts+=1
            if not all(compatible(h,value,k,v) for k,v in assignment.items()): continue
            new={k:list(v) for k,v in domains.items()}; feasible=True
            for k in hospitals:
                if k not in assignment and k!=h:
                    new[k]=[v for v in new[k] if compatible(h,value,k,v)]
                    if not new[k]: feasible=False; break
            if feasible:
                result=visit({**assignment,h:value},new)
                if result is not None: return result
        return None
    return visit({},make_domains()),attempts
csp_methods['Forward Checking + MRV']=forward_checking

### 4.3 Constraint Propagation (AC-3) + Backtracking
AC-3 removes a value if it has no compatible value in a neighboring variable’s domain. Requeue affected arcs until consistent, then search if necessary.

In [23]:
def ac3(domains):
    queue=deque((a,b) for a in hospitals for b in hospitals if a!=b)
    checks=0; removed=0
    while queue:
        a,b=queue.popleft(); kept=[]
        for x in domains[a]:
            supported=False
            for y in domains[b]:
                checks+=1
                if compatible(a,x,b,y): supported=True; break
            if supported: kept.append(x)
        if len(kept)<len(domains[a]):
            removed+=len(domains[a])-len(kept); domains[a]=kept
            if not kept: return False,checks,removed
            queue.extend((c,a) for c in hospitals if c not in (a,b))
    return True,checks,removed

def ac3_backtracking():
    domains=make_domains(); ok,checks,removed=ac3(domains)
    result,attempts=solve_backtracking(domains) if ok else (None,0)
    ac3_backtracking.details={'support_checks':checks,'values_removed':removed,'pruned_domains':domains}
    return result,attempts
csp_methods['AC-3 + Backtracking']=ac3_backtracking

### 4.4 Min-Conflicts
Start with a complete assignment; repeatedly pick a conflicted delivery and change its slot to minimize conflicts. Random restarts help escape plateaus. This is local search applied to a CSP.

In [24]:
def min_conflicts(seed=42,max_steps=100,restarts=10):
    rng=random.Random(seed); domains=make_domains(); changes=0
    if any(not d for d in domains.values()): return None,changes
    def conflicts(h,value,a):
        return sum(not compatible(h,value,k,v) for k,v in a.items() if k!=h)
    for _ in range(restarts):
        assignment={h:rng.choice(domains[h]) for h in hospitals}
        for _ in range(max_steps):
            bad=[h for h in hospitals if conflicts(h,assignment[h],assignment)]
            if not bad: return assignment,changes
            h=rng.choice(bad); scores={v:conflicts(h,v,assignment) for v in domains[h]}
            lowest=min(scores.values()); assignment[h]=rng.choice([v for v in scores if scores[v]==lowest]); changes+=1
    return None,changes
csp_methods['Min-Conflicts']=min_conflicts

In [25]:
csp_results=[]
for name,fn in csp_methods.items():
    t=time.perf_counter(); assignment,work=fn(); ms=(time.perf_counter()-t)*1000
    valid=assignment is not None and valid_assignment(assignment)
    csp_results.append((name,assignment,work,ms,valid))
    print(name,'| valid:',valid,'| work:',work,'| ms:',round(ms,4),'| slots:',assignment)
print('AC-3 inference details:',ac3_backtracking.details)
print('Work = attempted slot assignments for systematic solvers; repair moves for Min-Conflicts.')

Backtracking | valid: True | work: 13 | ms: 0.0257 | slots: {'Hospital North': 10, 'Hospital Central': 5, 'Hospital East': 0}
Forward Checking + MRV | valid: True | work: 3 | ms: 0.0173 | slots: {'Hospital East': 0, 'Hospital Central': 5, 'Hospital North': 10}
AC-3 + Backtracking | valid: True | work: 3 | ms: 0.0211 | slots: {'Hospital North': 10, 'Hospital Central': 5, 'Hospital East': 0}
Min-Conflicts | valid: True | work: 2 | ms: 0.0275 | slots: {'Hospital North': 10, 'Hospital Central': 5, 'Hospital East': 0}
AC-3 inference details: {'support_checks': 19, 'values_removed': 3, 'pruned_domains': {'Hospital North': [10], 'Hospital Central': [5], 'Hospital East': [0]}}
Work = attempted slot assignments for systematic solvers; repair moves for Min-Conflicts.


## 5. Verification and conclusions
Route correctness is checked against exhaustive simple paths on this tiny map. Minimum-minute methods are checked against the reference cost; minimum-road methods against the reference hop count. Local tours are checked for valid permutations and feasible cost. CSP results are checked against every domain and pair constraint.

Algorithms may tie on this small map. That does not mean they behave identically on larger problems. DFS/greedy/local methods do not promise the global minimum. Memory-bounded search depends on its budget. Min-Conflicts may fail within its limit despite an existing solution. Expand the dataset for stronger performance conclusions; simulated results are not operational emergency routing advice.

In [26]:
def all_simple_paths(start,goal):
    result=[]
    def visit(path):
        if path[-1]==goal: result.append(path); return
        for nxt in graph[path[-1]]:
            if nxt not in path: visit(path+[nxt])
    visit([start]); return result
paths=all_simple_paths(START,GOAL)
optimal_cost=min(map(path_cost,paths)); optimal_hops=min(len(x)-1 for x in paths)
for name,path,cost,_,_ in route_results:
    assert path and path[0]==START and path[-1]==GOAL
    assert all(b in graph[a] for a,b in zip(path,path[1:]))
    if name in ['UCS','A*','IDA*','RBFS','SMA* (20 records)']: assert cost==optimal_cost,name
    if name in ['BFS','IDDFS','Bidirectional BFS']: assert len(path)-1==optimal_hops,name
for name,tour,cost,_,_ in local_results:
    assert sorted(tour)==sorted(hospitals)
    assert cost>=tour_cost(reference)
for name,assignment,_,_,valid in csp_results: assert valid,name
assert depth_limited(START,GOAL,1)[0] is None
for name,fn in route_methods.items():
    path,_=fn(START,START); assert path==[START],name
print('All route, tour, CSP, shallow-limit and start=goal checks passed.')
print('Reference shortest time:',optimal_cost,'minutes; reference fewest roads:',optimal_hops)
print('Implemented:',len(route_methods),'route methods,',len(local_methods),'local methods,',len(csp_methods),'CSP configurations.')

All route, tour, CSP, shallow-limit and start=goal checks passed.
Reference shortest time: 13 minutes; reference fewest roads: 3
Implemented: 11 route methods, 4 local methods, 4 CSP configurations.


## 6. Viva explanation
- **Uninformed:** explore using graph structure or actual cost, without a goal-distance estimate.
- **Informed:** use a heuristic to guide exploration toward the destination.
- **Local:** modify candidate hospital visiting orders to improve their cost.
- **CSP:** assign dispatch slots while satisfying deadlines and loading-bay separation.

**Suggested report structure:** Aim → simulated data → four category headings → individual method headings → code/output → comparisons within each task → limitations → conclusion.